In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


csv저장 완료하고 데이터 로드부터 시작

In [2]:
# 프로모션 데이터 불러오기 (새로 추가된 코드)
promotion = pd.read_csv('./data/promotion_df.csv')
merged = pd.read_csv('./data/all_merged.csv')


아마 transactions_df는 따로 없는 df라 따로 저장을 해야하나??
merged 를 사용하면 될 듯

In [3]:
# 이번 프로모션 모든 매출의 합(총매출액)
total_revenue = merged['amount'].sum()
print(f"총 매출액: {total_revenue:,}$")

# transactions_df가 실제 결제(transaction)만 모아둔 표이고, 그 안의 amount가 각 결제 금액임

총 매출액: 1,775,451.97$


In [4]:
# 최종 인정된 행들의 amount만 합쳐서 프로모션 귀속 총매출액 계산
promo_total_revenue = promotion.loc[promotion['is_deduplicated'] == 1, 'amount'].sum()

print(f"[최종] 프로모션 귀속 총매출액: ${promo_total_revenue:,.2f}")
print(f"[최종] 귀속된 거래 수: {promotion['is_deduplicated'].sum():,}")
count_true_conversion = promotion['is_normal_flow'].sum()
print(f"is_normal_flow 합계: {count_true_conversion:,}")

[최종] 프로모션 귀속 총매출액: $442,993.24
[최종] 귀속된 거래 수: 22,333
is_normal_flow 합계: 23,267


In [5]:
# profit(순수익) 구하기
# profit 계산하기! (amount - reward_cost)
promotion['profit'] = promotion['amount'] - promotion['reward']
display(promotion.head())

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,age,became_member_on,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated,profit
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,33,2017-04-21,72000.0,8.57,1,1,1,0,0,3.57
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,33,2017-04-21,72000.0,14.11,1,1,1,0,0,12.11
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,33,2017-04-21,72000.0,10.27,1,0,1,0,0,8.27
3,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,discount,7,3,7,...,40,2018-01-09,57000.0,11.93,1,1,1,1,1,8.93
4,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,bogo,5,5,7,...,40,2018-01-09,57000.0,22.05,1,1,1,1,1,17.05


In [6]:
# 실제 reward 총지출액 (지출된 총 지출 비용)
total_reward_cost = promotion.loc[promotion['offer completed'].notna(), 'reward'].sum()

# ROI = 프로모션 투자(reward) 대비 수익(프로모션 총매출액)률
roi = (promo_total_revenue - total_reward_cost) / total_reward_cost * 100

print(f"프로모션 귀속 총 매출액: ${promo_total_revenue:,.2f}")
print(f"실제 reward 총지출액: ${total_reward_cost:,.2f}")
print(f"ROI: {roi:.2f}%")
print(f"실제 수익: {promo_total_revenue - total_reward_cost}")


프로모션 귀속 총 매출액: $442,993.24
실제 reward 총지출액: $162,426.00
ROI: 172.74%
실제 수익: 280567.24


In [7]:
# [유효 지출] 진짜 마케팅 퍼널(received->viewed->trans->comp)을 제대로 밟은 고객에게 쓴 돈
valid_reward_cost = promotion.loc[promotion['offer completed'].notna() & promotion['is_deduplicated'], 'reward'].sum()

# [낭비된 지출] 체리피커나 더블카운팅으로 허공에 날린 돈
wasted_reward_cost = total_reward_cost - valid_reward_cost

print(f"[재무 기준] 프로모션 총 리워드 비용: ${total_reward_cost:,.2f}")
print(f"[마케팅 기준] 광고로 유도된 '유효 지출': ${valid_reward_cost:,.2f} ({valid_reward_cost/total_reward_cost*100:.1f}%)")
print(f"[개선 필요] 체리피커 등에게 '낭비된 지출': ${wasted_reward_cost:,.2f} ({wasted_reward_cost/total_reward_cost*100:.1f}%)")

[재무 기준] 프로모션 총 리워드 비용: $162,426.00
[마케팅 기준] 광고로 유도된 '유효 지출': $109,742.00 (67.6%)
[개선 필요] 체리피커 등에게 '낭비된 지출': $52,684.00 (32.4%)


In [8]:
# 1. 분석을 위해 '순수 프로모션 기여 데이터'만 따로 떼어냅니다.
valid_promo_df = promotion[promotion['is_deduplicated'] == 1].copy()

print("[스타벅스 프로모션 심층 분석 리포트]\n")

# ==========================================
# 1️⃣ 프로모션 객단가 (AOV) 분석
# ==========================================
# 전체 결제 건의 객단가 vs 프로모션 결제 건의 객단가
total_aov = merged['amount'].mean()
promo_aov = valid_promo_df['amount'].mean()

print("1. 장바구니 크기(AOV) 효과")
print(f"   - 평소 평균 결제액 (자연 매출 객단가): ${total_aov:.2f}")
print(f"   - 쿠폰 사용 시 평균 결제액 (프로모션 객단가): ${promo_aov:.2f}")
if promo_aov > total_aov:
    print(f"인사이트: 쿠폰을 주면 고객들이 평소보다 ${promo_aov - total_aov:.2f} 만큼 더 씁니다! (업셀링 성공)\n")
else:
    print(f"인사이트: 쿠폰을 줘도 결제액이 늘지 않습니다. 구매 조건(허들)을 높여야 합니다.\n")

# ==========================================
# 2️⃣ BOGO vs Discount 맞짱 뜨기 (A/B Test)
# ==========================================
print("2. 프로모션 타입별 성과 (BOGO vs Discount)")
# 타입별로 묶어서 결제 건수와 매출액 합산
type_performance = valid_promo_df.groupby('offer_id').agg(
    결제건수=('person', 'count'),
    기여매출액=('amount', 'sum')
).reset_index()

for index, row in type_performance.iterrows():
    print(f"   - [{row['offer_id'].upper()}] 기여 결제: {row['결제건수']:,}건 / 기여 매출: ${row['기여매출액']:,.2f}")
print("인사이트: 두 프로모션 중 어떤 것이 회사에 더 많은 현금을 가져다주었는지 한눈에 비교됩니다.\n")


[스타벅스 프로모션 심층 분석 리포트]

1. 장바구니 크기(AOV) 효과
   - 평소 평균 결제액 (자연 매출 객단가): $12.78
   - 쿠폰 사용 시 평균 결제액 (프로모션 객단가): $19.84
인사이트: 쿠폰을 주면 고객들이 평소보다 $7.06 만큼 더 씁니다! (업셀링 성공)

2. 프로모션 타입별 성과 (BOGO vs Discount)
   - [BOGO_10_10_5] 기여 결제: 2,654건 / 기여 매출: $62,602.39
   - [BOGO_10_10_7] 기여 결제: 2,467건 / 기여 매출: $58,204.91
   - [BOGO_5_5_5] 기여 결제: 3,448건 / 기여 매출: $68,251.35
   - [BOGO_5_5_7] 기여 결제: 2,031건 / 기여 매출: $35,512.63
   - [DISCOUNT_10_2_10] 기여 결제: 4,375건 / 기여 매출: $78,095.81
   - [DISCOUNT_10_2_7] 기여 결제: 1,981건 / 기여 매출: $39,576.25
   - [DISCOUNT_20_5_10] 기여 결제: 1,147건 / 기여 매출: $30,280.09
   - [DISCOUNT_7_3_7] 기여 결제: 4,230건 / 기여 매출: $70,469.81
인사이트: 두 프로모션 중 어떤 것이 회사에 더 많은 현금을 가져다주었는지 한눈에 비교됩니다.



In [ ]:
# 기존 결제건수=('tx_key', 'count')였던 걸
# 기존 결제건수=('person', 'count')으로 바꿔도 되는 것은
# count는 값의 종류가 아니라 비어 있지 않은 행 수를 세기 때문이다.

In [9]:
# 쿠폰별 이탈률도 확인
offer_list = sorted(merged['offer_id'].dropna().unique())

for offer in offer_list:
    # 1) 해당 쿠폰의 received / viewed / completed만 모으기
    offer_raw_df = merged[merged['offer_id'] == offer].copy()

    step1_received = len(offer_raw_df[offer_raw_df['event'] == 'offer received'])
    step2_viewed = len(offer_raw_df[offer_raw_df['event'] == 'offer viewed'])

    # 2) promotion에서 해당 쿠폰만 필터링
    offer_final_df = promotion[promotion['offer_id'] == offer].copy()

    # 3) 조회 후 완료된 건수
    step3_completed = len(
        offer_final_df[offer_final_df['offer viewed'] <= offer_final_df['offer completed']]
    )

    # 4) 전환율 / 이탈률
    view_rate = (step2_viewed / step1_received) * 100 if step1_received > 0 else 0
    view_drop_rate = 100 - view_rate

    purchase_rate = (step3_completed / step2_viewed) * 100 if step2_viewed > 0 else 0
    purchase_drop_rate = 100 - purchase_rate

    total_cvr = (step3_completed / step1_received) * 100 if step1_received > 0 else 0

    print(f"=== {offer} 퍼널 전환율 및 이탈률 ===")
    print(f"1단계 (발송): {step1_received:,}건")
    print(f"2단계 (조회): {step2_viewed:,}건 (전환율: {view_rate:.1f}%, 이탈률: {view_drop_rate:.1f}%)")
    print(f"3단계 (결제): {step3_completed:,}건 (전환율: {purchase_rate:.1f}%, 이탈률: {purchase_drop_rate:.1f}%)")
    print(f"최종 전환율: {total_cvr:.1f}%")
    print("-" * 50)


=== bogo_10_10_5 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,593건
2단계 (조회): 7,298건 (전환율: 96.1%, 이탈률: 3.9%)
3단계 (결제): 2,739건 (전환율: 37.5%, 이탈률: 62.5%)
최종 전환율: 36.1%
--------------------------------------------------
=== bogo_10_10_7 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,658건
2단계 (조회): 6,716건 (전환율: 87.7%, 이탈률: 12.3%)
3단계 (결제): 2,582건 (전환율: 38.4%, 이탈률: 61.6%)
최종 전환율: 33.7%
--------------------------------------------------
=== bogo_5_5_5 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,571건
2단계 (조회): 7,264건 (전환율: 95.9%, 이탈률: 4.1%)
3단계 (결제): 3,514건 (전환율: 48.4%, 이탈률: 51.6%)
최종 전환율: 46.4%
--------------------------------------------------
=== bogo_5_5_7 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,677건
2단계 (조회): 4,171건 (전환율: 54.3%, 이탈률: 45.7%)
3단계 (결제): 2,106건 (전환율: 50.5%, 이탈률: 49.5%)
최종 전환율: 27.4%
--------------------------------------------------
=== discount_10_2_10 퍼널 전환율 및 이탈률 ===
1단계 (발송): 7,597건
2단계 (조회): 7,327건 (전환율: 96.4%, 이탈률: 3.6%)
3단계 (결제): 4,576건 (전환율: 62.5%, 이탈률: 37.5%)
최종 전환율: 60.2%
--------------------------------------------------
===

In [10]:
rows = []

for offer in offer_list:
    offer_raw_df = merged[merged['offer_id'] == offer].copy()
    offer_final_df = promotion[promotion['offer_id'] == offer].copy()

    step1_received = len(offer_raw_df[offer_raw_df['event'] == 'offer received'])
    step2_viewed = len(offer_raw_df[offer_raw_df['event'] == 'offer viewed'])
    step3_completed = len(
        offer_final_df[offer_final_df['offer viewed'] <= offer_final_df['offer completed']]
    )

    view_rate = (step2_viewed / step1_received) * 100 if step1_received > 0 else 0
    view_drop_rate = 100 - view_rate

    purchase_rate = (step3_completed / step2_viewed) * 100 if step2_viewed > 0 else 0
    purchase_drop_rate = 100 - purchase_rate

    total_cvr = (step3_completed / step1_received) * 100 if step1_received > 0 else 0

    rows.append({
        'offer_name': offer,
        'received': step1_received,
        'viewed': step2_viewed,
        'completed': step3_completed,
        'view_rate(%)': round(view_rate, 1),
        'view_drop_rate(%)': round(view_drop_rate, 1),
        'purchase_rate(%)': round(purchase_rate, 1),
        'purchase_drop_rate(%)': round(purchase_drop_rate, 1),
        'total_cvr(%)': round(total_cvr, 1),
    })

offer_funnel_df = pd.DataFrame(rows)

# 최종 전환율 내림차순으로 정렬해서 표시
display(offer_funnel_df.sort_values('total_cvr(%)', ascending=False))


,offer_name,received,viewed,completed,view_rate(%),view_drop_rate(%),purchase_rate(%),purchase_drop_rate(%),total_cvr(%)
4,discount_10_2_10,7597,7327,4576,96.4,3.6,62.5,37.5,60.2
7,discount_7_3_7,7646,7337,4348,96.0,4.0,59.3,40.7,56.9
2,bogo_5_5_5,7571,7264,3514,95.9,4.1,48.4,51.6,46.4
0,bogo_10_10_5,7593,7298,2739,96.1,3.9,37.5,62.5,36.1
1,bogo_10_10_7,7658,6716,2582,87.7,12.3,38.4,61.6,33.7
5,discount_10_2_7,7632,4118,2102,54.0,46.0,51.0,49.0,27.5
3,bogo_5_5_7,7677,4171,2106,54.3,45.7,50.5,49.5,27.4
6,discount_20_5_10,7668,2663,1300,34.7,65.3,48.8,51.2,17.0
8,informational_0_0_3,7618,6687,0,87.8,12.2,0.0,100.0,0.0
9,informational_0_0_4,7617,4144,0,54.4,45.6,0.0,100.0,0.0
